## COMP90024 Team 2



# Weather, Events, and Social Media Sentiment Across Australian Cities

**COMP90024 — Cluster and Cloud Computing — Assignment 2**

**Team \<2\>**

---

### Research Questions

1. Do weather conditions influence public sentiment on social media across Sydney, Melbourne and Brisbane?
2. How do major news events compare to weather as drivers of online sentiment?

### Data Sources

- Social media posts from Reddit, Mastodon and BlueSky (via the Fission REST API → ElasticSearch)
- Bureau of Meteorology (BOM) daily weather observations

### Scenarios

| # | Scenario | Key question | Primary API mode |
|---|---|---|---|
| 1 | City Activity Comparison | Which city generates the most social media activity? | `daily`, `summary` |
| 2 | City Sentiment Comparison | Do the three cities differ in overall sentiment? | `daily`, `summary`, `posts` |
| 3 | Temperature × Sentiment | Are people more negative on hotter days? | `daily`, `temp_buckets` |
| 4 | Rainfall × Sentiment | Does rain affect what people post? | `daily` |
| 5 | Hot vs Cool Day Language | Do people discuss different topics on hot and cool days? | `posts` (with `from`/`to`) |
| 6 | Event-Driven Sentiment | Do major news events drive sentiment more than weather? | `daily`, `posts` |

### Data Pipeline

All data is fetched via the RESTful API exposed by the Fission backend  
(`GET http://localhost:9090/api/query`), backed by ElasticSearch on the NeCTAR cloud.

Run before executing:

```bash
kubectl port-forward -n fission service/router 9090:80
```

**Available API modes:** `health`, `summary`, `daily`, `temp_buckets`, `posts`  
**Supported filters:** `city`, `platform`, `from` (date), `to` (date), `size`

Set `ANALYTICS_API_URL` to use a different endpoint. Analysis requires a populated
cleaned-post index with city/date, sentiment and matched weather fields. Empty
required windows stop with an explanatory message. API connectivity alone does
not establish harvester health or a causal weather effect.


---

## 0. Setup

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import rcParams
import seaborn as sns
from scipy.stats import pearsonr, spearmanr, mannwhitneyu
from wordcloud import WordCloud, STOPWORDS
from collections import Counter
from itertools import combinations
from datetime import datetime
import re


# ── Unified plot styling ─────────────────────────────────────
sns.set_theme(style='whitegrid', context='notebook',
              palette='muted', font_scale=1.0)

rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.edgecolor': '#333333',
    'axes.linewidth': 0.8,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.titlepad': 12,
    'axes.labelsize': 11,
    'axes.labelcolor': '#333333',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'xtick.color': '#555555',
    'ytick.color': '#555555',
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.frameon': False,
    'legend.fontsize': 10,
    'grid.color': '#E5E5E5',
    'grid.linewidth': 0.6,
    'figure.titlesize': 15,
    'figure.titleweight': 'bold',
    'savefig.dpi': 110,
    'figure.dpi': 100,
})

# ── API helper ───────────────────────────────────────────────
import os
API_BASE = os.environ.get("ANALYTICS_API_URL", "http://localhost:9090/api/query")

def api_get(**params):
    """Call the Fission REST API."""
    response = requests.get(API_BASE, params=params, timeout=120)
    response.raise_for_status()
    return response.json()

# ── Project palette ──────────────────────────────────────────
CITY_COLORS = {
    'sydney':    '#E07A1F',   # warm orange
    'melbourne': '#1F6FB4',   # deep blue
    'brisbane':  '#2E8B57',   # sea green
}
CITY_ORDER = ['sydney', 'melbourne', 'brisbane']

PLATFORM_COLORS = {
    'reddit':   '#FF4500',
    'mastodon': '#6364FF',
    'bluesky':  '#0085FF',
}

SENTIMENT_COLORS = {
    'positive': '#4CAF50',
    'neutral':  '#9E9E9E',
    'negative': '#E53935',
}

print(f'Notebook executed at: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

def require_frame(frame, columns, context):
    """Stop with actionable input requirements before constructing a plot."""
    missing = sorted(set(columns) - set(frame.columns))
    if frame.empty or missing:
        raise RuntimeError(
            f'{context}: no usable records or missing fields {missing}. '
            'Check API connectivity, the cleaned index, weather joins and date filters.'
        )
    return frame


### 0.1 Health check

Verify the Fission API and ElasticSearch are reachable.

In [ ]:
health = api_get(mode="health")
print(f"API status   : {health.get('status')}")
print(f"ES version   : {health.get('es_version')}")
print(f"ES cluster   : {health.get('cluster')}")

---

## 1. Load Data

We fetch data from four API endpoints, three of which return server-side aggregations covering the full ElasticSearch corpus:

| API mode | What it returns | Coverage |
|---|---|---|
| `mode=summary` | Total counts by city, platform, sentiment | Full corpus |
| `mode=daily` | Per-day aggregation (sentiment, temperature, rainfall) | Full corpus |
| `mode=temp_buckets` | Sentiment by temperature range | Full corpus |
| `mode=posts` | Raw post text (with `from`/`to` date filters) | Up to 1,000 per query |

In [ ]:
# ── 1.1 Summary (full dataset counts) ────────────────────────
summary = api_get(mode="summary")
if not sum(summary.get('by_city', {}).values()):
    raise RuntimeError('The cleaned-post index is empty. Run ingestion and cleaning before this analysis.')
print("Dataset Summary (full ElasticSearch corpus)")
print("=" * 55)
print(f"  Date range    : {summary['date_range'][0]} → {summary['date_range'][1]}")
print(f"  By city       : {summary['by_city']}")
print(f"  By platform   : {summary['by_platform']}")
print(f"  By sentiment  : {summary['by_sentiment']}")
total_posts = sum(summary['by_city'].values())
print(f"  Total posts   : {total_posts:,}")

In [ ]:
# ── 1.2 Daily aggregation ────────────────────
daily_frames = []
for city in CITY_ORDER:
    payload = api_get(mode="daily", city=city)
    rows = payload.get("days", [])
    df = pd.DataFrame(rows)
    require_frame(df, ['date', 'count', 'sentiment_mean', 'tmax_mean', 'prcp_mean'], f'Daily data for {city}')
    df['city'] = city
    daily_frames.append(df)
    print(f"  {city.capitalize():12s}  {len(rows):,} days of aggregated data")

daily = pd.concat(daily_frames, ignore_index=True)
daily['date'] = pd.to_datetime(daily['date'])
daily['year'] = daily['date'].dt.year
daily['month'] = daily['date'].dt.month

daily.rename(columns={
    'sentiment_mean': 'sent_mean',
    'tmax_mean': 'tmax',
    'prcp_mean': 'prcp',
    'count': 'n_posts'
}, inplace=True)

print(f"\nTotal daily records: {len(daily):,}")
print(f"Date range         : {daily['date'].min().date()} → {daily['date'].max().date()}")
daily.head()

In [ ]:
# ── 1.3 Temperature buckets  ──────────────────
temp_bucket_frames = []
for city in CITY_ORDER:
    payload = api_get(mode="temp_buckets", city=city)
    for b in payload.get("buckets", []):
        b['city'] = city
        temp_bucket_frames.append(b)

temp_buckets = pd.DataFrame(temp_bucket_frames)
require_frame(temp_buckets, ['city', 'bucket', 'count', 'sentiment_mean'], 'Temperature buckets')
print("Temperature bucket data (server-side aggregation on full corpus):\n")
print(temp_buckets.to_string(index=False))

In [ ]:
# ── 1.4 Raw posts for text analysis ──────────────────────────
# Fetch summer (Dec–Feb) and winter (Jun–Aug) windows across multiple years
# to ensure both hot and cool day text is available for Scenario 5.

summer_ranges = [
    ("2024-12-01", "2025-03-01"),
    ("2023-12-01", "2024-03-01"),
    ("2022-12-01", "2023-03-01"),
]

winter_ranges = [
    ("2025-06-01", "2025-08-31"),
    ("2024-06-01", "2024-08-31"),
    ("2023-06-01", "2023-08-31"),
]

post_frames = []

for city in CITY_ORDER:
    for start, end in summer_ranges + winter_ranges:
        payload = api_get(mode="posts", city=city, size=1000,
                         **{"from": start, "to": end})
        rows = payload.get("rows", [])
        if rows:
            post_frames.append(pd.DataFrame(rows))

if not post_frames:
    raise RuntimeError('No posts match the configured summer/winter windows. Adjust those dates to the available corpus.')
posts = pd.concat(post_frames, ignore_index=True)
require_frame(posts, ['id', 'city', 'platform', 'sentiment', 'text', 'tmax'], 'Seasonal post sample')
posts = posts.drop_duplicates(subset=['id'], keep='first')

posts['date_local'] = pd.to_datetime(
    posts.get('date_local', posts.get('created_local')), errors='coerce')
posts['sentiment'] = pd.to_numeric(posts['sentiment'], errors='coerce')
for col in ['tmax', 'prcp', 'tavg']:
    if col in posts.columns:
        posts[col] = pd.to_numeric(posts[col], errors='coerce')

posts['sent_label'] = pd.cut(
    posts['sentiment'],
    bins=[-1.01, -0.05, 0.05, 1.01],
    labels=['negative', 'neutral', 'positive']
)

print(f"Raw posts loaded   : {len(posts):,}")
print(f"Date range         : {posts['date_local'].min().date()} → {posts['date_local'].max().date()}\n")
print("Per city breakdown:")
for city in CITY_ORDER:
    sub = posts[posts['city'] == city]
    hot = len(sub[sub['tmax'] >= 32]) if 'tmax' in sub.columns else 0
    cool = len(sub[sub['tmax'] < 22]) if 'tmax' in sub.columns else 0
    print(f"  {city.capitalize():12s}  total={len(sub):,}  hot ≥32°C={hot}  cool <22°C={cool}")

### 1.5 Dataset Overview

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

# Panel 1: posts by city
city_counts = pd.Series(summary['by_city']).reindex(CITY_ORDER)
bars = axes[0].bar(
    [c.capitalize() for c in city_counts.index], city_counts.values,
    color=[CITY_COLORS[c] for c in city_counts.index],
    alpha=0.9, edgecolor='white', linewidth=1.2)
for bar, v in zip(bars, city_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 v + city_counts.max() * 0.015,
                 f'{v:,}', ha='center', fontweight='bold', fontsize=10)
axes[0].set_title('Posts by City')
axes[0].set_ylabel('Number of posts')
axes[0].margins(y=0.12)

# Panel 2: posts by platform
plat_counts = pd.Series(summary['by_platform']).sort_values(ascending=False)
bar_colors = [PLATFORM_COLORS.get(p, '#78909C') for p in plat_counts.index]
bars = axes[1].bar(
    plat_counts.index, plat_counts.values,
    color=bar_colors, alpha=0.9, edgecolor='white', linewidth=1.2)
for i, (idx, v) in enumerate(plat_counts.items()):
    axes[1].text(i, v + plat_counts.max() * 0.015,
                 f'{v:,}', ha='center', fontweight='bold', fontsize=10)
axes[1].set_title('Posts by Platform')
axes[1].set_ylabel('')
axes[1].margins(y=0.12)

# Panel 3: sentiment split
sent_counts = summary['by_sentiment']
axes[2].pie(
    [sent_counts.get('positive', 0),
     sent_counts.get('neutral', 0),
     sent_counts.get('negative', 0)],
    labels=['Positive', 'Neutral', 'Negative'],
    colors=[SENTIMENT_COLORS['positive'],
            SENTIMENT_COLORS['neutral'],
            SENTIMENT_COLORS['negative']],
    autopct='%1.1f%%', startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    textprops={'fontsize': 11})
axes[2].set_title('Overall Sentiment Split')

plt.suptitle('Dataset Overview — Full ElasticSearch Corpus', y=1.02)
plt.tight_layout()
plt.show()

weighted_mean = (daily['sent_mean'] * daily['n_posts']).sum() / daily['n_posts'].sum()
print(f'\nWeighted mean sentiment across the corpus: {weighted_mean:+.3f}')

---

## 2. Scenario 1 — City Activity Comparison

**Question.** Which Australian city generates the most social media activity?

**Method.** Daily post volume from `mode=daily` (full dataset, server-side aggregation).

In [ ]:
# ── Daily post volume time series ────────────────────────────
fig, ax = plt.subplots(figsize=(13, 4.5))
for city in CITY_ORDER:
    sub = daily[daily['city'] == city].sort_values('date')
    ax.plot(sub['date'], sub['n_posts'],
            marker='o', markersize=2, linewidth=1.2,
            color=CITY_COLORS[city], label=city.capitalize(), alpha=0.85)

ax.set_title('Daily Post Volume by City')
ax.set_ylabel('Posts per day')
ax.legend(loc='upper left')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

print('City activity summary (full dataset):')
print('-' * 55)
for city in CITY_ORDER:
    sub = daily[daily['city'] == city]
    print(f'  {city.capitalize():12s}  '
          f'mean={sub["n_posts"].mean():>5.1f}/day  '
          f'max={sub["n_posts"].max():>5d}  '
          f'total={sub["n_posts"].sum():>9,}  '
          f'days={len(sub):,}')

In [ ]:
# ── Yearly volume by city ────────────────────────────────────
yearly_city = (daily.groupby(['year', 'city'])['n_posts']
               .sum().unstack(fill_value=0)
               .reindex(columns=CITY_ORDER))

fig, ax = plt.subplots(figsize=(11, 4.5))
yearly_city.plot(
    kind='bar', ax=ax,
    color=[CITY_COLORS[c] for c in CITY_ORDER],
    alpha=0.9, edgecolor='white', linewidth=0.8, width=0.8)
ax.set_title('Yearly Post Volume by City')
ax.set_ylabel('Number of posts')
ax.set_xlabel('')
ax.legend([c.capitalize() for c in CITY_ORDER], title='', loc='upper left')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

print('Yearly totals (all cities combined):')
print('-' * 35)
for year in sorted(daily['year'].unique()):
    total = daily[daily['year'] == year]['n_posts'].sum()
    print(f'  {year}: {total:>9,} posts')

---

## 3. Scenario 2 — City Sentiment Comparison

**Question.** Do Sydney, Melbourne and Brisbane differ in overall sentiment?

**Method.** Weighted mean sentiment from `mode=daily` (full dataset) plus distribution from `mode=posts`.

In [ ]:
# ── Weighted mean sentiment per city (full data) ─────────────
print('Weighted mean sentiment per city (full dataset):')
print('=' * 60)
city_weighted = {}
for city in CITY_ORDER:
    sub = daily[daily['city'] == city].dropna(subset=['sent_mean'])
    weighted = (sub['sent_mean'] * sub['n_posts']).sum() / sub['n_posts'].sum()
    total_posts_city = sub['n_posts'].sum()
    city_weighted[city] = weighted
    print(f'  {city.capitalize():12s}  '
          f'mean={weighted:+.3f}  '
          f'n_days={len(sub):>5,}  '
          f'total_posts={total_posts_city:>9,}')

In [ ]:
# ── Distribution + mean ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: violin plot from sample
sns.violinplot(
    data=posts, x='city', y='sentiment', hue='city', legend=False, dodge=False,
    order=CITY_ORDER, ax=axes[0],
    palette=CITY_COLORS, alpha=0.8, inner='quartile', cut=0)
axes[0].axhline(0, color='#888888', linewidth=0.6, linestyle=':')
axes[0].set_title('Sentiment Distribution (sample)')
axes[0].set_xlabel('')
axes[0].set_ylabel('VADER compound score')
axes[0].set_xticks(range(len(CITY_ORDER)))
axes[0].set_xticklabels([c.capitalize() for c in CITY_ORDER])

# Right: weighted means from full data
city_se = {}
for city in CITY_ORDER:
    sub = daily[daily['city'] == city].dropna(subset=['sent_mean'])
    weights = sub['n_posts'].to_numpy(dtype=float)
    values = sub['sent_mean'].to_numpy(dtype=float)
    mean = np.average(values, weights=weights)
    effective_n = weights.sum() ** 2 / np.square(weights).sum()
    city_se[city] = np.sqrt(np.average((values - mean) ** 2, weights=weights) / effective_n)

bars = axes[1].bar(
    [c.capitalize() for c in city_weighted.keys()],
    list(city_weighted.values()),
    yerr=list(city_se.values()),
    color=[CITY_COLORS[c] for c in city_weighted.keys()],
    alpha=0.9, capsize=6, edgecolor='white', linewidth=1.2)
for bar, v in zip(bars, city_weighted.values()):
    axes[1].text(bar.get_x() + bar.get_width()/2, v + 0.005,
                 f'{v:+.3f}', ha='center',
                 fontweight='bold', fontsize=11)
axes[1].axhline(0, color='#888888', linewidth=0.6, linestyle=':')
axes[1].set_title('Weighted Mean ± Descriptive SE (daily samples)')
axes[1].set_ylabel('Mean sentiment')
axes[1].set_xlabel('')
axes[1].margins(y=0.15)

plt.tight_layout()
plt.show()

In [ ]:
# ── Mann-Whitney U tests on daily means (full data) ──────────
print('Pairwise Mann-Whitney U tests on daily sentiment means:')
print('-' * 60)
for c1, c2 in combinations(CITY_ORDER, 2):
    s1 = daily[daily['city'] == c1]['sent_mean'].dropna()
    s2 = daily[daily['city'] == c2]['sent_mean'].dropna()
    if len(s1) > 5 and len(s2) > 5:
        stat, p = mannwhitneyu(s1, s2, alternative='two-sided')
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        print(f'  {c1.capitalize()} vs {c2.capitalize():<10s}  '
              f'U={stat:>11,.0f}  p={p:.4f}  {sig}')

In [ ]:
# ── Sentiment by platform ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4.5))
sns.boxplot(
    data=posts, x='platform', y='sentiment', hue='platform', legend=False, dodge=False, ax=ax,
    palette=PLATFORM_COLORS,
    order=['reddit', 'mastodon', 'bluesky'],
    showfliers=False, width=0.6)
ax.axhline(0, color='#888888', linewidth=0.6, linestyle=':')
ax.set_title('Sentiment Distribution by Platform')
ax.set_xlabel('')
ax.set_ylabel('VADER compound score')
plt.tight_layout()
plt.show()

print('Mean sentiment per platform (sample):')
print('-' * 40)
for plat in sorted(posts['platform'].unique()):
    grp = posts[posts['platform'] == plat]
    print(f'  {plat:12s}  mean={grp["sentiment"].mean():+.3f}  n={len(grp):,}')

In [ ]:
# ── Yearly sentiment trend per city (full data) ──────────────
yearly_sent = daily.groupby(['year', 'city'])[['sent_mean', 'n_posts']].apply(
    lambda g: (g['sent_mean'] * g['n_posts']).sum() / g['n_posts'].sum()
).unstack().reindex(columns=CITY_ORDER)

fig, ax = plt.subplots(figsize=(11, 4.5))
for city in CITY_ORDER:
    if city in yearly_sent.columns:
        ax.plot(yearly_sent.index, yearly_sent[city],
                marker='o', color=CITY_COLORS[city],
                linewidth=2.2, markersize=7, label=city.capitalize())
ax.axhline(0, color='#888888', linewidth=0.6, linestyle=':')
ax.set_title('Yearly Sentiment Trend by City (full data)')
ax.set_ylabel('Weighted mean sentiment')
ax.set_xlabel('Year')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

---

## 4. Scenario 3 — Temperature × Sentiment

**Question.** Are people more negative on social media when it's hotter?

**Method.** Pearson and Spearman correlations from `mode=daily` and a temperature-bucket analysis from `mode=temp_buckets`. Both use the full dataset.

In [ ]:
# ── Per-city scatter with regression ─────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), sharey=True)

for ax, city in zip(axes, CITY_ORDER):
    sub = daily[daily['city'] == city].dropna(subset=['tmax', 'sent_mean'])
    if len(sub) < 3 or sub['tmax'].nunique() < 2:
        ax.set_title(f'{city.capitalize()} — insufficient data')
        continue

    ax.scatter(sub['tmax'], sub['sent_mean'],
               c=sub['tmax'], cmap='RdYlBu_r',
               s=sub['n_posts'].clip(upper=200) * 1.4,
               alpha=0.65, edgecolors='white', linewidth=0.4)

    z = np.polyfit(sub['tmax'], sub['sent_mean'], 1)
    p_line = np.poly1d(z)
    x_range = np.linspace(sub['tmax'].min(), sub['tmax'].max(), 50)
    ax.plot(x_range, p_line(x_range),
            color='#C62828', linestyle='--', linewidth=2, alpha=0.8)

    pr, pp = pearsonr(sub['tmax'], sub['sent_mean'])
    sig = '***' if pp < 0.001 else '**' if pp < 0.01 else '*' if pp < 0.05 else 'ns'
    ax.set_title(f'{city.capitalize()}\nr = {pr:+.3f}, p = {pp:.3f} ({sig})')
    ax.set_xlabel('Max temperature (°C)')
    ax.axhline(0, color='#888888', linewidth=0.6, linestyle=':')

axes[0].set_ylabel('Mean daily sentiment')
plt.suptitle('Temperature vs. Sentiment — per City (full data)', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation summary table ────────────────────────────────
print('Temperature × Sentiment Correlation per City (Full Data)')
print('=' * 70)
print(f'{"City":12s}  {"N days":>7s}  {"Pearson r":>10s}  {"p-value":>9s}  '
      f'{"Spearman ρ":>10s}  {"p-value":>9s}  Sig.')
print('-' * 70)

for city in CITY_ORDER:
    sub = daily[daily['city'] == city].dropna(subset=['sent_mean', 'tmax'])
    if len(sub) < 5:
        print(f'{city.capitalize():12s}  {len(sub):>7d}  — not enough data —')
        continue
    pr, pp = pearsonr(sub['tmax'], sub['sent_mean'])
    sr, sp = spearmanr(sub['tmax'], sub['sent_mean'])
    sig = '***' if pp < 0.001 else '**' if pp < 0.01 else '*' if pp < 0.05 else ''
    print(f'{city.capitalize():12s}  {len(sub):>7,}  '
          f'{pr:+10.3f}  {pp:>9.4f}  '
          f'{sr:+10.3f}  {sp:>9.4f}  {sig}')

print('\nSignificance: * p<0.05  ** p<0.01  *** p<0.001')

In [ ]:
# ── Temperature-bucket analysis (server-side, full corpus) ──
bucket_order = ['cold (<18)', 'mild (18-24)', 'warm (24-30)',
                'hot (30-35)', 'extreme (>=35)']
bucket_labels = ['Cold\n(<18°C)', 'Mild\n(18-24)', 'Warm\n(24-30)',
                 'Hot\n(30-35)', 'Extreme\n(≥35°C)']
bucket_palette = ['#4575B4', '#91BFDB', '#FEE090', '#FC8D59', '#D73027']

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), sharey=True)

for ax, city in zip(axes, CITY_ORDER):
    city_data = (temp_buckets[temp_buckets['city'] == city]
                 .set_index('bucket').reindex(bucket_order))

    bars = ax.bar(bucket_labels, city_data['sentiment_mean'],
                  color=bucket_palette,
                  alpha=0.92, edgecolor='white', linewidth=1.2)

    for bar, (_, row) in zip(bars, city_data.iterrows()):
        height = bar.get_height()
        if not pd.isna(height):
            offset = 0.005 if height >= 0 else -0.012
            ax.text(bar.get_x() + bar.get_width()/2,
                    height + offset,
                    f'{height:+.3f}',
                    ha='center', fontsize=9, fontweight='bold')
            ax.text(bar.get_x() + bar.get_width()/2,
                    -0.04,
                    f'n={int(row["count"]):,}',
                    ha='center', fontsize=8, color='#666666')

    ax.axhline(0, color='#888888', linewidth=0.6, linestyle=':')
    ax.set_title(city.capitalize())
    ax.set_xlabel('')
    ax.tick_params(axis='x', labelsize=9)

axes[0].set_ylabel('Mean sentiment')
plt.suptitle('Sentiment by Temperature Range — Server-Side Aggregation', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Filtered correlation (days with ≥10 posts) ───────────────
agg_filtered = daily[daily['n_posts'] >= 10].copy()

fig, ax = plt.subplots(figsize=(11, 5))
for city in CITY_ORDER:
    sub = agg_filtered[agg_filtered['city'] == city].dropna(subset=['tmax', 'sent_mean'])
    ax.scatter(sub['tmax'], sub['sent_mean'],
               color=CITY_COLORS[city], alpha=0.45, s=35,
               label=city.capitalize(), edgecolor='white', linewidth=0.3)
    if len(sub) > 10 and sub['tmax'].nunique() > 1:
        z = np.polyfit(sub['tmax'], sub['sent_mean'], 1)
        p_line = np.poly1d(z)
        x_range = np.linspace(sub['tmax'].min(), sub['tmax'].max(), 50)
        ax.plot(x_range, p_line(x_range),
                color=CITY_COLORS[city], linestyle='--',
                linewidth=2, alpha=0.85)

ax.axhline(0, color='#888888', linewidth=0.6, linestyle=':')
ax.set_xlabel('Max temperature (°C)')
ax.set_ylabel('Mean daily sentiment')
ax.set_title('Temperature vs Sentiment — High-Volume Days Only (≥10 posts)')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

print('Correlation on high-volume days only (≥10 posts):')
print('-' * 60)
for city in CITY_ORDER:
    sub = agg_filtered[agg_filtered['city'] == city].dropna(subset=['tmax', 'sent_mean'])
    if len(sub) >= 10:
        pr, pp = pearsonr(sub['tmax'], sub['sent_mean'])
        sig = '***' if pp < 0.001 else '**' if pp < 0.01 else '*' if pp < 0.05 else 'ns'
        print(f'  {city.capitalize():12s}  n={len(sub):>4,}  '
              f'r={pr:+.3f}  p={pp:.4f}  {sig}')

---

## 5. Scenario 4 — Rainfall × Sentiment

**Question.** Does rain make people more negative on social media?

**Method.** Compare rainy (precipitation > 0 mm) vs dry days using `mode=daily` (full dataset).

In [ ]:
# ── Rainy vs dry day analysis ────────────────────────────────
daily['is_rainy'] = daily['prcp'] > 0

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

rain_data = daily.dropna(subset=['prcp', 'sent_mean'])
sns.boxplot(
    data=rain_data, x='city', y='sent_mean',
    hue='is_rainy', order=CITY_ORDER, ax=axes[0],
    palette={False: '#FF8F00', True: '#1F6FB4'},
    showfliers=False, width=0.7)
axes[0].axhline(0, color='#888888', linewidth=0.6, linestyle=':')
axes[0].set_title('Sentiment: Rainy vs Dry Days')
axes[0].set_xlabel('')
axes[0].set_ylabel('Mean daily sentiment')
axes[0].set_xticks(range(len(CITY_ORDER)))
axes[0].set_xticklabels([c.capitalize() for c in CITY_ORDER])
handles = axes[0].get_legend_handles_labels()[0]
axes[0].legend(handles, ['Dry', 'Rainy'], title='Day type', loc='lower right')

rain_volume = (rain_data.groupby(['city', 'is_rainy'])['n_posts']
               .sum().unstack(fill_value=0).reindex(index=CITY_ORDER, columns=[False, True], fill_value=0))
rain_volume.columns = ['Dry', 'Rainy']
rain_volume.plot(
    kind='bar', ax=axes[1],
    color=['#FF8F00', '#1F6FB4'], alpha=0.9,
    edgecolor='white', linewidth=0.8, width=0.7)
axes[1].set_title('Post Volume: Rainy vs Dry Days')
axes[1].set_ylabel('Number of posts')
axes[1].set_xlabel('')
axes[1].set_xticklabels([c.capitalize() for c in CITY_ORDER], rotation=0)
axes[1].legend(title='', loc='upper right')

plt.tight_layout()
plt.show()

print('Rainy vs Dry Day Sentiment Comparison (full data):')
print('=' * 70)
for city in CITY_ORDER:
    sub = rain_data[rain_data['city'] == city]
    rainy = sub[sub['is_rainy']]['sent_mean']
    dry = sub[~sub['is_rainy']]['sent_mean']
    if len(rainy) > 5 and len(dry) > 5:
        stat, p = mannwhitneyu(rainy, dry, alternative='two-sided')
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        print(f'  {city.capitalize():12s}  '
              f'rainy={rainy.mean():+.3f} (n={len(rainy)})  '
              f'dry={dry.mean():+.3f} (n={len(dry)})  '
              f'p={p:.4f}  {sig}')

In [ ]:
# ── Rainfall intensity analysis ──────────────────────────────
daily['rain_level'] = pd.cut(
    daily['prcp'],
    bins=[-0.01, 0, 2, 10, 100],
    labels=['No Rain', 'Light\n(0-2 mm)',
            'Moderate\n(2-10 mm)', 'Heavy\n(>10 mm)'])

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), sharey=True)
rain_order = ['No Rain', 'Light\n(0-2 mm)', 'Moderate\n(2-10 mm)', 'Heavy\n(>10 mm)']
rain_palette = ['#FF8F00', '#90CAF9', '#1F6FB4', '#0D47A1']

for ax, city in zip(axes, CITY_ORDER):
    sub = daily[daily['city'] == city].dropna(subset=['rain_level'])
    sns.boxplot(
        data=sub, x='rain_level', y='sent_mean', hue='rain_level', hue_order=rain_order, legend=False, dodge=False,
        order=rain_order, ax=ax,
        palette=rain_palette, showfliers=False, width=0.6)
    ax.axhline(0, color='#888888', linewidth=0.6, linestyle=':')
    ax.set_title(city.capitalize())
    ax.set_xlabel('')

axes[0].set_ylabel('Mean daily sentiment')
plt.suptitle('Sentiment by Rainfall Intensity (full data)', y=1.02)
plt.tight_layout()
plt.show()

---

## 6. Scenario 5 — Hot-Day vs Cool-Day Language

**Question.** Do people discuss different topics on hot days versus cool days?

**Method.** Word clouds and log-odds ratio on post text fetched via `mode=posts` with `from`/`to` date filters covering both summer (Dec–Feb) and winter (Jun–Aug).

In [ ]:
# ── Word clouds for each city ────────────────────────────────
stop = set(STOPWORDS)
stop.update({
    'sydney', 'melbourne', 'brisbane', 'amp', 'http', 'https',
    'com', 'www', 'just', 'like', 'know', 'got', 'one', 'get',
    'really', 'would', 'also', 'much', 'going', 'still', 'even',
    'well', 'back', 'right', 'think', 'good', 'time', 'people',
    'make', 'way', 'want', 'need', 'thing', 'say', 'see', 'look',
    'take', 'come', 'day', 'year', 'new', 'use', 'will', 'said',
    'removed', 'deleted', 'nan'
})

def make_wordcloud(text, title, ax, colormap='viridis'):
    if len(text) < 200:
        ax.text(0.5, 0.5, 'Not enough text',
                ha='center', va='center', fontsize=14, color='#999999')
        ax.set_title(title)
        ax.axis('off')
        return
    wc = WordCloud(
        width=900, height=450, stopwords=stop,
        background_color='white', collocations=False,
        colormap=colormap, max_words=60, relative_scaling=0.5
    )
    frequencies = wc.process_text(text)
    if not frequencies:
        ax.text(0.5, 0.5, 'No usable words', ha='center', va='center')
        ax.set_title(title)
        ax.axis('off')
        return
    wc.generate_from_frequencies(frequencies)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(title)
    ax.axis('off')

for city in CITY_ORDER:
    sub = posts[(posts['city'] == city) & posts['tmax'].notna()]
    hot_posts = sub[sub['tmax'] >= 32]
    cool_posts = sub[sub['tmax'] < 22]

    hot_text = ' '.join(hot_posts['text'].astype(str).tolist())
    cool_text = ' '.join(cool_posts['text'].astype(str).tolist())

    fig, axes = plt.subplots(1, 2, figsize=(15, 4))
    make_wordcloud(hot_text,
                   f'{city.capitalize()} — Hot days ≥32°C  (n={len(hot_posts)})',
                   axes[0], 'YlOrRd')
    make_wordcloud(cool_text,
                   f'{city.capitalize()} — Cool days <22°C  (n={len(cool_posts)})',
                   axes[1], 'YlGnBu')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Top words side-by-side ───────────────────────────────────
def top_words(texts, n=15):
    all_words = []
    for t in texts:
        words = re.findall(r'[a-z]{3,}', str(t).lower())
        all_words.extend([w for w in words if w not in stop])
    return Counter(all_words).most_common(n)

print('Top 15 Words: Hot Days vs Cool Days')
print('=' * 72)
for city in CITY_ORDER:
    sub = posts[(posts['city'] == city) & posts['tmax'].notna()]
    hot_p = sub[sub['tmax'] >= 32]['text'].tolist()
    cool_p = sub[sub['tmax'] < 22]['text'].tolist()

    print(f'\n{city.capitalize()}  ({len(hot_p)} hot / {len(cool_p)} cool posts)')
    print('-' * 72)

    hot_top = top_words(hot_p)
    cool_top = top_words(cool_p)

    print(f'  {"Rank":>4s}  {"Hot day word":<20s} {"Count":>6s}   '
          f'{"Cool day word":<20s} {"Count":>6s}')
    for i in range(min(15, max(len(hot_top), len(cool_top)))):
        hw, hc = hot_top[i] if i < len(hot_top) else ('—', '')
        cw, cc = cool_top[i] if i < len(cool_top) else ('—', '')
        print(f'  {i+1:>4d}  {hw:<20s} {str(hc):>6s}   {cw:<20s} {str(cc):>6s}')

In [ ]:
# ── Distinctive words: log-odds ratio ────────────────────────
def distinctive_words(texts_a, texts_b, stop_words, top_n=15):
    def word_freq(texts):
        counter = Counter()
        for t in texts:
            words = re.findall(r'[a-z]{3,}', str(t).lower())
            counter.update([w for w in words if w not in stop_words and w != 'removed'])
        return counter

    freq_a = word_freq(texts_a)
    freq_b = word_freq(texts_b)
    all_words = set(freq_a.keys()) | set(freq_b.keys())
    total_a = sum(freq_a.values()) + 1
    total_b = sum(freq_b.values()) + 1
    scores = {}
    for word in all_words:
        a_rate = (freq_a.get(word, 0) + 1) / total_a
        b_rate = (freq_b.get(word, 0) + 1) / total_b
        scores[word] = np.log2(a_rate / b_rate)
    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_scores[:top_n], sorted_scores[-top_n:][::-1]

print('Words DISTINCTIVELY associated with Hot vs Cool days')
print('Log-odds ratio (positive = more common on hot days)')
print('=' * 72)

for city in CITY_ORDER:
    sub = posts[(posts['city'] == city) & posts['tmax'].notna()]
    hot_p = sub[sub['tmax'] >= 32]['text'].tolist()
    cool_p = sub[sub['tmax'] < 22]['text'].tolist()

    if len(hot_p) < 20 or len(cool_p) < 20:
        print(f'\n{city.capitalize()}: not enough posts '
              f'(hot={len(hot_p)}, cool={len(cool_p)})')
        continue

    top_hot, top_cool = distinctive_words(hot_p, cool_p, stop)

    print(f'\n{city.capitalize()}  ({len(hot_p)} hot / {len(cool_p)} cool)')
    print(f'  {"Hot-day words":<32s}  {"Cool-day words":<32s}')
    print('  ' + '-' * 70)
    for i in range(min(15, len(top_hot), len(top_cool))):
        hw, hs = top_hot[i]
        cw, cs = top_cool[i]
        print(f'  {hw:<20s} ({hs:+.2f})   {cw:<20s} ({cs:+.2f})')

---

## 7. Scenario 6 — Event-Driven Sentiment: Bondi Beach Shooting (December 2025)

**Question.** Do major news events affect social media sentiment more than weather?

**Method.** Analyse three cities' reactions to the Bondi Beach shooting (14–15 Dec 2025).  
This scenario tests whether **geographic proximity to an event** drives sentiment more than ambient weather conditions.

**Context.** On 14 December 2025, a shooting incident occurred at Bondi Beach, Sydney.  
We examine the three weeks of 13 Dec 2025 – 2 Jan 2026 across all three cities.

In [ ]:
# ── Fetch daily data for the event window ───────────────────
event_start = "2025-12-13"
event_end   = "2026-01-02"
event_date  = "2025-12-14"

event_daily = {}
for city in CITY_ORDER:
    payload = api_get(mode="daily", city=city,
                      **{"from": event_start, "to": event_end})
    df = pd.DataFrame(payload.get("days", []))
    require_frame(df, ['date', 'count', 'sentiment_mean', 'tmax_mean', 'prcp_mean'], f'Event window for {city}')
    df['date'] = pd.to_datetime(df['date'])
    df.rename(columns={'count': 'n_posts', 'sentiment_mean': 'sent_mean',
                       'tmax_mean': 'tmax', 'prcp_mean': 'prcp'}, inplace=True)
    df['city'] = city
    event_daily[city] = df
    print(f'  {city.capitalize():12s}  {len(df)} days loaded')

print(f'\nObservation window: {event_start} → {event_end}')

In [ ]:
# ── Sentiment + volume timeline ──────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                          gridspec_kw={'height_ratios': [2, 1]})

# Top: sentiment
for city in CITY_ORDER:
    sub = event_daily[city].sort_values('date')
    axes[0].plot(sub['date'], sub['sent_mean'],
                 marker='o', color=CITY_COLORS[city],
                 linewidth=2.2, markersize=6, label=city.capitalize())

axes[0].axhline(0, color='#888888', linewidth=0.6, linestyle=':')
axes[0].axvspan(pd.Timestamp('2025-12-14'), pd.Timestamp('2025-12-15'),
                alpha=0.18, color='#D32F2F', label='Shooting incident')
axes[0].set_ylabel('Mean daily sentiment')
axes[0].set_title('Bondi Beach Shooting — City Sentiment Response')
axes[0].legend(loc='lower right')

# Bottom: volume
for city in CITY_ORDER:
    sub = event_daily[city].sort_values('date')
    axes[1].plot(sub['date'], sub['n_posts'],
                 marker='s', color=CITY_COLORS[city], linestyle='--',
                 linewidth=1.8, markersize=5, label=city.capitalize())

axes[1].axvspan(pd.Timestamp('2025-12-14'), pd.Timestamp('2025-12-15'),
                alpha=0.18, color='#D32F2F')
axes[1].set_ylabel('Posts per day')
axes[1].set_xlabel('Date')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.xticks(rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# ── Phase summary table ──────────────────────────────────────
print('Sentiment Before vs During the Event')
print('=' * 75)
print(f'{"City":12s}  {"Before (Dec 13)":>16s}  {"Peak (Dec 14-15)":>16s}  '
      f'{"Δ":>8s}  {"Posts (peak)":>12s}')
print('-' * 75)

for city in CITY_ORDER:
    sub = event_daily[city]
    before = sub[sub['date'] == '2025-12-13']['sent_mean'].values
    peak_mask = sub['date'].isin(pd.to_datetime(['2025-12-14', '2025-12-15']))
    peak = sub[peak_mask]['sent_mean'].mean()
    peak_posts = sub[peak_mask]['n_posts'].sum()

    before_val = before[0] if len(before) > 0 else np.nan
    change = peak - before_val if not np.isnan(before_val) else np.nan

    print(f'{city.capitalize():12s}  {before_val:+16.3f}  {peak:+16.3f}  '
          f'{change:+8.3f}  {peak_posts:>12,}')

In [ ]:
# ── Phased bar chart: before / event / after ────────────────
phases = {
    'Before\n(Dec 13)':       ['2025-12-13'],
    'Event\n(Dec 14-15)':     ['2025-12-14', '2025-12-15'],
    'Recovery\n(Dec 18-20)':  ['2025-12-18', '2025-12-19', '2025-12-20'],
    'Restored\n(Dec 28-31)':  ['2025-12-28', '2025-12-29', '2025-12-30', '2025-12-31'],
}

fig, ax = plt.subplots(figsize=(13, 5))

x = np.arange(len(phases))
width = 0.26
offsets = {'sydney': -width, 'melbourne': 0, 'brisbane': width}

for city in CITY_ORDER:
    sub = event_daily[city]
    vals = []
    for phase_name, dates in phases.items():
        mask = sub['date'].isin(pd.to_datetime(dates))
        vals.append(sub[mask]['sent_mean'].mean())

    bars = ax.bar(x + offsets[city], vals, width,
                  label=city.capitalize(), color=CITY_COLORS[city],
                  alpha=0.9, edgecolor='white', linewidth=1.2)
    for bar, v in zip(bars, vals):
        if not np.isnan(v):
            offset = 0.008 if v >= 0 else -0.018
            ax.text(bar.get_x() + bar.get_width()/2, v + offset,
                    f'{v:+.3f}', ha='center',
                    fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(list(phases.keys()))
ax.axhline(0, color='#888888', linewidth=0.6, linestyle=':')
ax.set_ylabel('Mean sentiment')
ax.set_title('Sentiment by Phase: Before / Event / Recovery / Restored')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# ── Fetch post text for the event period ────────────────────
syd_event_payload = api_get(mode="posts", city="sydney", size=1000,
                             **{"from": "2025-12-14", "to": "2025-12-15"})
syd_event_posts = pd.DataFrame(syd_event_payload.get("rows", []))

melb_event_payload = api_get(mode="posts", city="melbourne", size=1000,
                              **{"from": "2025-12-14", "to": "2025-12-15"})
melb_event_posts = pd.DataFrame(melb_event_payload.get("rows", []))

bris_event_payload = api_get(mode="posts", city="brisbane", size=1000,
                              **{"from": "2025-12-14", "to": "2025-12-15"})
bris_event_posts = pd.DataFrame(bris_event_payload.get("rows", []))

print(f'Sydney during event    : {len(syd_event_posts)} posts')
print(f'Melbourne during event : {len(melb_event_posts)} posts')
print(f'Brisbane during event  : {len(bris_event_posts)} posts')

In [ ]:
# ── Word clouds: three cities during the event ──────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

texts = {
    'sydney': ' '.join(syd_event_posts.get('text', pd.Series(dtype=str)).dropna().astype(str).tolist()),
    'melbourne': ' '.join(melb_event_posts.get('text', pd.Series(dtype=str)).dropna().astype(str).tolist()),
    'brisbane': ' '.join(bris_event_posts.get('text', pd.Series(dtype=str)).dropna().astype(str).tolist()),
}
counts = {
    'sydney': len(syd_event_posts),
    'melbourne': len(melb_event_posts),
    'brisbane': len(bris_event_posts),
}
cmaps = {'sydney': 'Reds', 'melbourne': 'Blues', 'brisbane': 'Greens'}

for ax, city in zip(axes, CITY_ORDER):
    if len(texts[city]) > 200:
        wc = WordCloud(width=900, height=450, stopwords=stop,
                       background_color='white', collocations=False,
                       colormap=cmaps[city], max_words=60)
        frequencies = wc.process_text(texts[city])
        if frequencies:
            wc.generate_from_frequencies(frequencies)
            ax.imshow(wc, interpolation='bilinear')
        else:
            ax.text(0.5, 0.5, 'No usable words', ha='center', va='center')
    else:
        ax.text(0.5, 0.5, 'Not enough text', ha='center', va='center')
    ax.set_title(f'{city.capitalize()} (n={counts[city]})')
    ax.axis('off')

plt.suptitle('What were people talking about on 14–15 Dec 2025?', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Distinctive words: Sydney vs Melbourne (same window) ────
if len(syd_event_posts) > 20 and len(melb_event_posts) > 20:
    syd_texts = syd_event_posts.get('text', pd.Series(dtype=str)).dropna().tolist()
    melb_texts = melb_event_posts.get('text', pd.Series(dtype=str)).dropna().tolist()

    top_syd, top_melb = distinctive_words(syd_texts, melb_texts, stop)

    print('Words DISTINCTIVELY associated with each city on 14-15 Dec 2025')
    print('Log-odds ratio (positive = more common in Sydney)')
    print('=' * 72)
    print(f'  {"Sydney (event city)":<32s}  {"Melbourne (control)":<32s}')
    print('  ' + '-' * 70)
    for i in range(min(15, len(top_syd), len(top_melb))):
        sw, ss = top_syd[i]
        mw, ms = top_melb[i]
        print(f'  {sw:<20s} ({ss:+.2f})   {mw:<20s} ({ms:+.2f})')
else:
    print('Not enough posts for log-odds comparison.')

# Also compare Sydney vs Brisbane
if len(syd_event_posts) > 20 and len(bris_event_posts) > 20:
    syd_texts = syd_event_posts.get('text', pd.Series(dtype=str)).dropna().tolist()
    bris_texts = bris_event_posts.get('text', pd.Series(dtype=str)).dropna().tolist()
    top_syd2, top_bris = distinctive_words(syd_texts, bris_texts, stop)

    print('\n\nSydney vs Brisbane (same dates)')
    print('=' * 72)
    print(f'  {"Sydney (event city)":<32s}  {"Brisbane (control)":<32s}')
    print('  ' + '-' * 70)
    for i in range(min(15, len(top_syd2), len(top_bris))):
        sw, ss = top_syd2[i]
        bw, bs = top_bris[i]
        print(f'  {sw:<20s} ({ss:+.2f})   {bw:<20s} ({bs:+.2f})')

In [ ]:
# ── Sydney recovery curve ────────────────────────────────────
syd_full = event_daily['sydney'].sort_values('date').reset_index(drop=True)
baseline_values = syd_full.loc[syd_full['date'] == '2025-12-13', 'sent_mean'].dropna()
if baseline_values.empty:
    raise RuntimeError('The event comparison needs a valid Sydney baseline on 2025-12-13. Check the date range.')
baseline = baseline_values.iloc[0]

fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(syd_full['date'], syd_full['sent_mean'],
        marker='o', color=CITY_COLORS['sydney'],
        linewidth=2.4, markersize=7)
ax.axhline(baseline, color='#888888', linewidth=1.2, linestyle='--',
           label=f'Pre-event baseline ({baseline:+.3f})')
ax.axhline(0, color='#888888', linewidth=0.5, linestyle=':')
ax.axvspan(pd.Timestamp('2025-12-14'), pd.Timestamp('2025-12-15'),
           alpha=0.18, color='#D32F2F', label='Shooting incident')

# Annotate every 2nd point to keep it readable
for i, row in syd_full.iterrows():
    if i % 2 == 0:
        ax.annotate(f'{row["sent_mean"]:+.3f}',
                    xy=(row['date'], row['sent_mean']),
                    textcoords="offset points", xytext=(0, 12),
                    ha='center', fontsize=8, fontweight='bold')

ax.set_ylabel('Mean daily sentiment')
ax.set_xlabel('Date')
ax.set_title('Sydney Sentiment Recovery After Bondi Shooting')
ax.legend(loc='lower right')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

# Recovery analysis
recovery_day = None
for _, row in syd_full.iterrows():
    if row['date'] > pd.Timestamp('2025-12-15') and row['sent_mean'] > 0:
        recovery_day = row['date'].strftime('%Y-%m-%d')
        break

if recovery_day:
    days_to_recover = (pd.Timestamp(recovery_day) - pd.Timestamp(event_date)).days
    print(f'Sydney sentiment first turned positive on {recovery_day} '
          f'({days_to_recover} days after the incident).')
    baseline_recovery = syd_full[(syd_full['date'] > pd.Timestamp('2025-12-15')) & (syd_full['sent_mean'] >= baseline)]
    if not baseline_recovery.empty:
        print(f'Pre-event baseline first reached on {baseline_recovery.iloc[0]["date"].date()}.')
else:
    print('Sydney sentiment did not return to positive within the observation window.')

---

## 8. Discussion

### Summary of Scenarios

| Scenario | Question | Method | Data Source |
|---|---|---|---|
| 1. City Activity | Which city is most active? | Daily post counts | `mode=daily` (full corpus) |
| 2. City Sentiment | Do cities differ in sentiment? | Weighted means + Mann-Whitney | `mode=daily` (full corpus) |
| 3. Temperature × Sentiment | Does heat affect sentiment? | Pearson/Spearman + `temp_buckets` | `mode=daily` + `mode=temp_buckets` |
| 4. Rainfall × Sentiment | Does rain affect sentiment? | Rainy vs dry comparison | `mode=daily` (full corpus) |
| 5. Hot/Cool Language | Different topics by temperature? | Word cloud + log-odds | `mode=posts` (summer/winter) |
| 6. Event-Driven Sentiment | Do events drive sentiment more than weather? | Pre/event/recovery comparison | `mode=daily` + `mode=posts` |

### Historical snapshot observations

The following figures were recorded in the original project analysis. They are
not recomputed by this Markdown cell and should not be treated as the output
of a new run. The current API may contain a different corpus.

### Recorded comparison

**Major news events drive sentiment far more powerfully than weather conditions.**

| Factor | Effect on sentiment |
|---|---|
| Extreme heat (≥35 °C) | Δ ≈ −0.03 (Scenario 3) |
| Heavy rain (>10 mm) | Δ ≈ −0.01 (Scenario 4) |
| Bondi shooting (Dec 2025) | **Δ ≈ −0.38** (Scenario 6) |

The event effect is **10–30× stronger** than any weather effect found in Scenarios 3–4.

### Additional Observations

- **Geographic proximity matters.** Sydney's sentiment plunged after the shooting; Melbourne dipped only slightly; Brisbane was essentially unaffected.
- **Volume as an event detector.** Sydney's post count surged 28× on the day of the incident — post volume itself is a useful event signal.
- **Slow recovery.** Sydney's sentiment did not return to its pre-event baseline for roughly 15 days.
- **Implication for Scenario 5.** "Distinctive hot-day words" such as *shooting* or *firearms* in our log-odds analysis were not weather-driven — they were artefacts of specific events that happened to co-occur with summer.

### Limitations

- **Scenario 5 sampling.** Post text comes from `mode=posts`, which returns up to 1,000 records per query. We mitigate this by querying multiple summer/winter date ranges, but the text sample remains smaller than the full corpus.
- **Cleaned vs raw data.** All posts in ElasticSearch carry a `cleaned_at` timestamp, indicating they passed the cleaning pipeline. The corpus has grown beyond the original parquet snapshot as harvesters continue collecting.
- **Correlation is not causation.** News events, weekday effects, and seasonal confounds are not controlled for in the weather analyses.
- **VADER limitations.** Lexicon-based sentiment analysis can miss sarcasm, irony and Australian slang.
- **Platform evolution.** BlueSky and Mastodon contributions are concentrated from 2023 onward, affecting longitudinal interpretation pre-/post-2023.

### Data Pipeline

All results in this notebook are fetched via the Fission REST API (`http://localhost:9090/api/query`).  
Scenarios 1–4 use **server-side ElasticSearch aggregations** (`mode=daily`, `mode=temp_buckets`, `mode=summary`) covering the full corpus.  
Scenarios 5–6 use **`mode=posts` with `from`/`to` date filters** to fetch raw text for targeted windows.

---
*Data pipeline: Jupyter Notebook → Fission REST API → ElasticSearch (NeCTAR MRC).*